

<img src="images/strathsdr_banner.png" align="left" >

# RFSoC OFDM Transceiver
----

<div class="alert alert-box alert-info">
Please use Jupyter Labs http://board_ip_address/lab for this notebook.
</div>

This notebook demonstrates the implementation of an Orthogonal Frequency Division Multiplexing (OFDM) transceiver on RFSoC. PYNQ is used to control the underlying modulation scheme of the OFDM sub-carriers and for visualisation of data at various stages in the transmit/receive chain, such as the received constellations.       

## Aims 
* To demonstrate a complete OFDM transceiver.
* Explain the various stages OFDM comprises.
* Provide an interactive and responsive means to inspect the data at each stage.

## Table of Contents
* [Introduction](#introduction)
    * [Hardware Setup](#hardware-setup)
    * [Software Setup](#software-setup)
* [Transmit](#transmit)
    * [Symbol Generation](#symbol-gen)
* [Receive](#creating-images)
    * [Constellation Plot](#constellation-plot)
* [Conclusion](#conclusion)

## References
* [Xilinx, Inc, "USP RF Data Converter: LogiCORE IP Product Guide", PG269, v2.4, November 2020](https://www.xilinx.com/support/documentation/ip_documentation/usp_rf_data_converter/v2_4/pg269-rf-data-converter.pdf)

## Revision History
* **v1.0** | 26/02/2021 | OFDM transceiver notebook
* **v1.1** | 30/03/2022 | OFDM DUC and DDC change
* **v1.2** | 17/05/2023 | General notebook for all boards.

----

## Introduction <a class="anchor" id="introduction"></a>

The demonstrator is a complete OFDM transceiver. This notebook will explain each stage of the system with a combination of text, diagrams and live data capture. [Figure 1](#fig-1) below provides an overview of the system.

<a class="anchor" id="fig-1"></a>
<figure>
<img src="./images/system_overview.png" height='75%' width='75%'/>
    <figcaption><b>Figure 1: OFDM Demonstrator System Overview</b></figcaption>
</figure>

The OFDM system starts with generation of random data symbols from 1 of 10 possible modulation schemes (BPSK to 1024 QAM), based on input provided from PYNQ. In accordance with the procedure used in the IEEE 802.11a/g standard, the symbols are grouped into blocks of 48 for mapping to sub-carriers. The OFDM symbol consists of 48 data sub-carriers, 4 pilot sub-carriers and 12 null sub-carriers (including DC). The final OFDM symbol is created by performing a 64 point IFFT and adding a 16 sample Cyclic Prefix (CP). The transmitted signal consists of a continuous stream of OFDM symbols. At the very beginning of the data stream, the L-STF and L-LTF training symbols from the IEEE 802.11a/g standard are transmitted, to aid synchronisation and channel estimation tasks in the receiver. 

In the receiver, timing and frequency synchronisation are performed to acquire symbol timing and correct for any frequency offsets. Once timing is achieved, the FFT is performed to recover the underlying data symbols in each OFDM symbol. The L-LTF symbols are used to estimate the channel frequency response at each sub-carrier position, which subsequently allows the data to be equalised. Finally, the pilot symbols are used to correct for residual phase errors in the phase tracking stage. The recovered symbols are then passed into the PS for visualisation in PYNQ.

### Hardware Setup <a class="anchor" id="hardware-setup"></a>
Your RFSoC development board should be setup in single channel mode with a loopback cable connected between an ADC and DAC.

See the setup instructions [here](01_rfsoc_ofdm_setup.ipynb) for more information.

<div class="alert alert-box alert-danger">
<b>Caution:</b>
    In this demonstration, we generate tones using the RFSoC development board. Your device should be setup in loopback mode. You should understand that the RFSoC platform can also transmit RF signals wirelessly. Remember that unlicensed wireless transmission of RF signals may be illegal in your geographical location. Radio signals may also interfere with nearby devices, such as pacemakers and emergency radio equipment. Note that it is also illegal to intercept and decode particular RF signals. If you are unsure, please seek professional support.
</div>

### Software Setup
The setup for the OFDM demonstration system is nearly complete. The majority of the libraries used by the demonstrator design are contained inside the RFSoC-OFDM software package. We only need to run a few code cells to initialise the environment.

In [1]:
# from pynq import PL
# PL.reset()

In [2]:
from rfsoc_ofdm.overlay import Overlay
# from pynq import Overlay
import ipywidgets as ipw

ofdm_hw = Overlay("/home/xilinx/jupyter_notebooks/rfsoc_ofdm/rfsoc_ofdm/notebooks/rfsoc_ofdm_evm15_tx2_rx3.bit", init_rf_clks=True, clks=409)



/home/xilinx/jupyter_notebooks/rfsoc_ofdm/rfsoc_ofdm/notebooks/rfsoc_ofdm_evm15_tx2_rx3.bit


/home/xilinx/jupyter_notebooks/rfsoc_ofdm/rfsoc_ofdm
LMX2594_384.00.txt
got to else...
LMX2594_491.52.txt
LMK04828_245.76.txt
LMX2594_409.6.txt
/home/xilinx/jupyter_notebooks/rfsoc_ofdm/rfsoc_ofdm/xrfclk/lmk04828/LMX2594_409.6.txt
/home/xilinx/jupyter_notebooks/rfsoc_ofdm/rfsoc_ofdm/xrfclk/lmk04828/LMK04828_245.76.txt
/home/xilinx/jupyter_notebooks/rfsoc_ofdm/rfsoc_ofdm/xrfclk/lmk04828/LMX2594_409.6.txt
LMK04828_245.76.txt
LMX2594_409.6.txt
Configuring adc and dac to: pll_freq=409.6
inside adcs
configure_adcs done
inside dacs
InvSincFIR =  2
mixer mode c2r =  2
configure_dacs done
['InspectorConstellation', 'InspectorConstellation/axi_dma', 'InspectorConstellation/data_inspector_module', 'InspectorEvm', 'InspectorEvm/axi_dma', 'InspectorEvm/axis_switch', 'InspectorEvm/data_inspector_module', 'InspectorReceiver', 'InspectorReceiver/axi_dma', 'InspectorReceiver/axis_switch', 'InspectorReceiver/data_inspector_module', 'InspectorTransmitter', 'InspectorTransmitter/axi_dma', 'InspectorTrans

In [3]:
mix = (ofdm_hw)
dir(mix)
# (ofdm_hw.dac_block.UpdateEvent(1))

['InspectorConstellation',
 'InspectorConstellation/axi_dma',
 'InspectorConstellation/data_inspector_module',
 'InspectorEvm',
 'InspectorEvm/axi_dma',
 'InspectorEvm/axis_switch',
 'InspectorEvm/data_inspector_module',
 'InspectorReceiver',
 'InspectorReceiver/axi_dma',
 'InspectorReceiver/axis_switch',
 'InspectorReceiver/data_inspector_module',
 'InspectorTransmitter',
 'InspectorTransmitter/axi_dma',
 'InspectorTransmitter/axis_switch',
 'InspectorTransmitter/data_inspector_module',
 'PSDDR',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_deepcopy_dict_from',
 '_ip_map',
 '_register_drivers',
 'adc_block',
 'adc_tile',
 'axi_intc',
 'binfile_nam

----

## Transmit <a class="anchor" id="transmit"></a>

### Symbol Generation <a class="anchor" id="symbol-gen"></a>
There are a total of 10 modulation schemes available to be transmitted. These are generated on the programmable logic and can be chosen between by updating the *ofdm_tx IP core's* **mod** register with a value from 0-9 over AXI4-Lite. [Figure 2](#fig-2) illustrates the IP core as a simplified block diagram.

<a class="anchor" id="fig-2"></a>
<figure>
<img src="./images/symbol_generation.png" height='45%' width='45%'/>
    <figcaption><b>Figure 2: Symbol generation block diagram.</b></figcaption>
</figure>


This drop down widget sends the value associated with each modulation scheme to the ofdm_tx core. Run the cell to use it.

In [4]:
ofdm_hw.ofdm_transmitter.modulation_dropdown.get_widget()

Dropdown(description='Modulation Scheme: ', layout=Layout(width='300px'), options=('BPSK', 'QPSK', '8-PSK', '1…

The output of the symbol generation block has been tapped off, allowing for the live symbols to be visualised in Jupyter Lab. Run the cell below and hit play on the chart to inspect the symbols generated on the programmable logic.

In [5]:
ipw.VBox([ofdm_hw.inspectors['transmitter'].time_plot(),
          ipw.HBox([ofdm_hw.inspectors['transmitter'].channel_widget.get_widget(),
                    
                    ofdm_hw.inspectors['transmitter'].plot_control()])])

    'data': [{'name': 'Real Signal',
              'type': 'scatter',
          …

---

## Receive

### Constellation Plot <a class="anchor" id="constellation-plot"></a>

Run the cell below to see the output of the OFDM receiver displayed as a constellation: 

In [6]:
ipw.VBox([ofdm_hw.inspectors['constellation'].constellation_plot(),
          ofdm_hw.inspectors['constellation'].plot_control()])

    'data': [{'mode': 'markers',
              'type': 'scatter',
              …

In [7]:
# ipw.VBox([ofdm_hw.inspectors['evm'].evm_plot(),
#           ofdm_hw.inspectors['evm'].plot_control()])

In [8]:
ipw.VBox([ofdm_hw.inspectors['receiver'].spectrum_plot(), 
          ipw.HBox([ofdm_hw.inspectors['receiver'].channel_widget.get_widget(),
                    # ofdm_hw.inspectors['receiver'].set_frequency(590),
                    ofdm_hw.inspectors['receiver'].plot_control()])])

    'data': [{'name': 'IQ Spectrum',
              'showlegend': True,
         …

In [9]:
ofdm_hw.ofdm_loopback_application()
# print(ofdm_hw.adc_block.BlockStatus)
# print(ofdm_hw.adc_block.QMCSettings)
# print(ofdm_hw.adc_block.PwrMode)
# # ofdm_hw.adc_tile.Reset()
# dir(ofdm_hw.adc_tile.Reset)

[-1.03411865-0.41851807j -0.4319458 -0.75891113j  0.15093994+0.746521j
  0.76623535+0.15185547j  0.16131592+0.1484375j  -1.05187988-0.46057129j
  0.7399292 -0.45043945j  0.44952393+0.16107178j -0.45178223+1.0446167j
  1.04333496-0.13800049j -0.4552002 +0.13653564j  1.04284668+0.14282227j
 -0.76739502-0.14935303j -0.15356445+0.44598389j -1.065979  -0.13806152j
 -0.4498291 -0.13842773j  0.15057373+1.04943848j  0.1605835 +0.16113281j
  0.14892578-0.15338135j -0.71868896-0.74957275j  0.46466064-0.44689941j
  0.43041992+0.12866211j  0.14672852+0.46136475j -1.03765869-0.46881104j
  0.45812988+0.73565674j  0.76281738-0.43988037j -0.44989014-0.46295166j
  1.05297852-0.44775391j -1.05432129+0.74578857j -0.74645996-0.47424316j
  0.13818359+0.44696045j  0.16882324-0.7432251j  -0.74945068-1.05328369j
  1.0557251 -0.42218018j  0.44329834-0.16491699j  0.46075439+0.45306396j
 -0.14990234-0.45092773j  0.46856689+0.75512695j -1.05285645-0.13067627j
  0.45202637-0.15777588j -1.04986572-1.0703125j  -0.75

    'data': [{'name': 'IQ Spectrum',…

0x26CC 0x0000 0x1702 0x0000 0x31E0 0x0000 0x325D 0x0000 0x15FD 0x0000 0x29B9 0x0000 0x1EBB 0x0000 0x2995 0x0000 0x13F3 0x0000 0x1FBB 0x0000 0x24BA 0x0000 0x26BB 0x0000 0x1983 0x0000 0x2ABC 0x0000 0x1F1D 0x0000 0x28C6 0x0000 0x2252 0x0000 0x1AB2 0x0000 0x17AE 0x0000 0x1B0A 0x0000 0x1B4A 0x0000 0x2895 0x0000 0x2B43 0x0000 0x027D 0x0000 0x15D1 0x0000 0x3222 0x0000 0x1700 0x0000 0x1155 0x0000 0x02CA 0x0000 0x159E 0x0000 0x0FF0 0x0000 0x1BA1 0x0000 0x1FCA 0x0000 0x324A 0x0000 0x16E7 0x0000 0x1B4F 0x0000 0x24C0 0x0000 0x03FA 0x0000 0x1BC4 0x0000 0x31BE 0x0000 0x2905 0x0000 0x175A 0x0000 0x10C0 0x0000 0x27D9 0x0000 0x22FE 0x0000 0x18AF 0x0000 0x15F5 0x0000 0x1BC6 0x0000 0x1A97 0x0000 0x2032 0x0000 0x03B5 0x0000 0x1054 0x0000 0x16AF 0x0000 0x29F6 0x0000 0x101F 0x0000 0x128D 0x0000 0x03D2 0x0000 0x034A 0x0000 0x2BFF 0x0000 0x2914 0x0000 0x2A7B 0x0000 0x1A67 0x0000 0x14D7 0x0000 0x1AA3 0x0000 0x102B 0x0000 0x2953 0x0000 0x26A2 0x0000 0x1727 0x0000 0x333C 0x0000 0x0EFC 0x0000 0x1EAF 0x0000 0x11A4

In [10]:
sw_mmio = ofdm_hw.axis_switch.mmio

CTRL_OFFSET     = 0x000
MI0_MUX_OFFSET  = 0x040
REG_UPDATE_MASK = 0x00000002  # bit 1

def axis_switch_select_input(si_index):
    """
    si_index = 0 -> connect S00 to M00
    si_index = 1 -> connect S01 to M00
    """
    # 1) Program MI0_MUX with desired slave index
    sw_mmio.write(MI0_MUX_OFFSET, si_index & 0xF)

    # 2) Trigger REG_UPDATE so the change takes effect
    ctrl_val = sw_mmio.read(CTRL_OFFSET)
    sw_mmio.write(CTRL_OFFSET, ctrl_val | REG_UPDATE_MASK)

[ 0.43505859+0.7487793j  -0.15863037+1.05322266j -1.0369873 -1.06982422j
 -1.0848999 +1.01782227j -1.03509521+1.04351807j -1.04272461+0.44866943j
  0.75640869+0.14532471j -0.45257568-0.74511719j  0.74645996+0.44873047j
  1.02606201+0.73480225j  0.13793945+0.75488281j -0.4487915 -0.15893555j
  0.14007568-1.05621338j -0.43713379+1.05627441j  0.44726562-0.75378418j
  0.14807129+1.05328369j  0.76293945-0.13098145j -0.12689209-0.44830322j
 -0.45465088-0.44665527j  0.44580078-0.45037842j  0.74121094-0.74627686j
  0.74261475+0.16564941j -0.144104  +0.75274658j  0.17047119-0.75561523j
  1.06262207-0.73480225j  0.76568604+0.7510376j  -0.45196533-0.14190674j
  0.45001221+0.4420166j  -1.02294922+1.06207275j -0.76501465-0.72576904j
 -0.14709473+0.45062256j  0.46466064-0.46350098j -0.73321533+1.04748535j
  0.4442749 +0.13525391j  0.15234375-0.76660156j  0.74664307-0.75524902j
  0.14038086+0.14471436j -0.14715576+0.13555908j -0.14758301+0.44372559j
 -0.74926758+0.14202881j -0.46289062-0.45495605j  0

AttributeError: Could not find IP or hierarchy axis_switch in overlay

In [ ]:
# Route input 0 -> output
# axis_switch_select_input(1)
ofdm_hw.dac_block.InvSincFIR = 1
print(ofdm_hw.dac_block.InvSincFIR)
dir(ofdm_hw.dac_block.InvSincFIR)

# Later, swap to input 1 -> output
# axis_switch_select_input(1)

In [ ]:
# from rfsoc_ofdm.overlay import Overlay
import numpy as np

# ol = Overlay("rfsoc_ofdm_csv.bit")

player   = ofdm_hw.axis_player          # /axis_player/s_axil
bram_ctl = ofdm_hw.axi_bram_ctrl        # /axi_bram_ctrl/S_AXI  (name may be axi_bram_ctrl_0)
sw_mmio  = ofdm_hw.axis_switch.mmio     # /axis_switch/S_AXI_CTRL

Try changing the modulation scheme using the code cell that was ran earlier. You should be able to visualise the modulation schemes in the plot above.

In [ ]:
csv_path = "/home/xilinx/jupyter_notebooks/rfsoc_ofdm/rfsoc_ofdm/notebooks/DDDDU_5G_id100_ssb120k_pdsch_1_2x_test_20ms.csv"
print("starting loading...")
# Load as float32, shape (N, 2): col 0 = I, col 1 = Q
# iq = np.loadtxt(csv_path, delimiter=',', dtype=np.float32)  # shape (N,2)
print("finished loading...")
fs = 240e6
N=8192
t = np.arange(N) /fs
I = np.sin(2*np.pi*10e6*t)
Q = 0

# Convert to signed Q1.15
scale = 1 << 15
I_fix = np.clip(I, -0.9999695, 0.9999695)
Q_fix = np.clip(Q, -0.9999695, 0.9999695)

I_fix = np.round(I_fix * scale).astype(np.int16)
Q_fix = np.round(Q_fix * scale).astype(np.int16)

# Pack into 32-bit words: [31:16]=I, [15:0]=Q
I32 = I_fix.astype(np.int32)
Q32 = Q_fix.astype(np.int32)
data_words = (I32 << 16) | (Q32 & 0xFFFF)

n_words = len(data_words)
print("Loaded", n_words, "complex samples")

In [ ]:

# Sanity: see how much BRAM MMIO range we have
print("BRAM MMIO length (bytes):", bram_ctl.mmio.length)

for i, word in enumerate(data_words):
    
    bram_ctl.mmio.write(i * 4, int(word))



In [ ]:
player_mmio = player.mmio  # or use player.register_map if you like

CTRL_OFFSET       = 0x00
LENGTH_OFFSET     = 0x04
STATUS_OFFSET     = 0x08
START_ADDR_OFFSET = 0x0C

CTRL_ENABLE = 1 << 0
CTRL_START  = 1 << 1
CTRL_REPEAT = 1 << 2

In [ ]:
def axis_player_start(n_words, start_word=0, repeat=False):
    # 1) Program start address & length
    player_mmio.write(START_ADDR_OFFSET, start_word)
    player_mmio.write(LENGTH_OFFSET,    n_words)

    # 2) Build control word (enable + optional repeat, start=0)
    ctrl = CTRL_ENABLE
    if repeat:
        ctrl |= CTRL_REPEAT

    # Ensure start bit is 0 first
    player_mmio.write(CTRL_OFFSET, ctrl)

    # 3) Now assert start bit (0->1 edge generates start_pulse)
    player_mmio.write(CTRL_OFFSET, ctrl | CTRL_START)

    # (you don't *have* to clear start afterwards; the FSM
    # just looks for the rising edge)

In [ ]:
axis_player_start(n_words, start_word=0, repeat=True)   # continuous loop
# or
axis_player_start(n_words, start_word=0, repeat=False)  # play once

## Conclusion <a class="anchor" id="conclusion"></a>
This notebook has demonstrated a live OFDM transceiver operating on an RFSoC. It has been shown how PYNQ can be used to interact with various parts of the hardware design, offering control of the system and visualisation of data. 

* The various components comprising the OFDM transmitter and receiver have been introduced at a high level. 
    * Modulation symbols were inspected.
    * The received and sychronised constellation were plotted.
* Interacted with a real-time RF system.
    * Changed modulation scheme.

----

⬅️ [Previous Notebook](01_rfsoc_ofdm_setup.ipynb) | [Next Notebook](03_voila_rfsoc_ofdm_demonstrator.ipynb) 🚀

----
----